# Predicting NBA Success Based on Rookie Stats

This notebook represents my workflow and data science techniques in building a sufficent model for our target.

Author: Rasheed Martin

In [ ]:
df_players_rookie_awards = df_players_rookie_awards.with_columns(
    pl.when(pl.col("Draft_Pick").is_null()).then(1000).otherwise(pl.col("Draft_Pick")).alias("Draft_Pick"),
    pl.when(pl.col("Draft_Class").is_null()).then(0).otherwise(1).alias("Was_Drafted"),
    pl.when(pl.col("All_Rookie_Rank").is_null()).then(pl.lit("No Votes")).otherwise(pl.col("All_Rookie_Rank")).alias("All_Rookie_Rank"),
    pl.when(pl.col("ROY_Rank").is_null()).then(0).otherwise(pl.col("ROY_Rank")).alias("ROY_Rank"),
    pl.when(pl.col("ROY_Votes").is_null()).then(0).otherwise(pl.col("ROY_Votes")).alias("ROY_Votes"),
    pl.when(pl.col("ROY_Point_Share").is_null()).then(0).otherwise(pl.col("ROY_Point_Share")).alias("ROY_Point_Share"),
    pl.when(pl.col("All_Rookie_First_Place_Votes").is_null()).then(0).otherwise(pl.col("All_Rookie_First_Place_Votes")).alias("All_Rookie_First_Place_Votes"),
    pl.when(pl.col("All_Rookie_Second_Place_Votes").is_null()).then(0).otherwise(pl.col("All_Rookie_Second_Place_Votes")).alias("All_Rookie_Second_Place_Votes"),
    pl.when(pl.col("All_Rookie_Point_Share").is_null()).then(0).otherwise(pl.col("All_Rookie_Point_Share")).alias("All_Rookie_Point_Share")
)
df_players_rookie_awards

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(20, 18))
sns.heatmap(rookie_stats_corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)

plt.title("Correlation Matrix")

# Offset tick positions by 0.5 to center labels under cells
plt.xticks(ticks=np.arange(len(rookie_stats_corr_matrix.columns)) + 0.5, 
           labels=rookie_stats_corr_matrix.columns, rotation=45, ha='right')

plt.yticks(ticks=np.arange(len(rookie_stats_corr_matrix.columns)) + 0.5, 
           labels=rookie_stats_corr_matrix.columns, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(20, 18))
mask = np.triu(np.ones_like(rookie_stats_corr_matrix, dtype=bool))
sns.heatmap(rookie_stats_corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True, mask=mask)

plt.title("Correlation Matrix")

# Offset tick positions by 0.5 to center labels under cells
plt.xticks(ticks=np.arange(len(rookie_stats_corr_matrix.columns)) + 0.5, 
           labels=rookie_stats_corr_matrix.columns, rotation=45, ha='right')

plt.yticks(ticks=np.arange(len(rookie_stats_corr_matrix.columns)) + 0.5, 
           labels=rookie_stats_corr_matrix.columns, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Convert the correlation matrix to long format using unpivot
long_rookie_stats = rookie_stats_corr_matrix.with_columns([
    pl.Series("feature_1", rookie_stats_corr_matrix.columns)
]).unpivot(index=["feature_1"], variable_name="feature_2", value_name="correlation")

# Remove self-correlations
long_rookie_stats = long_rookie_stats.filter(pl.col("feature_1") != pl.col("feature_2"))

long_rookie_stats = long_rookie_stats.with_columns([
    pl.when(pl.col("feature_1") < pl.col("feature_2"))
      .then(pl.col("feature_1") + "||" + pl.col("feature_2"))
      .otherwise(pl.col("feature_2") + "||" + pl.col("feature_1"))
      .alias("pair")
]).unique(subset=["pair"])

# Add absolute correlation and sort
long_rookie_stats = long_rookie_stats.with_columns([
    pl.col("correlation").abs().alias("abs_correlation")
]).sort("abs_correlation", descending=True)



long_rookie_stats.head(10)